# Summary Tables

This notebook generates summary tables from the cardiovascular association results and gnomAD frequency data. Each summary is saved as a separate sheet in a single Excel workbook.

In [16]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

associations_path = Path("../data/final/association_results.csv")
gnomad_path = Path("../data/final/gnomad_frequencies.csv")

out_dir = Path("../data/final")
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx = out_dir / "summary_tables.xlsx"

In [17]:
associations_df = pd.read_csv(associations_path, sep=";", low_memory=False)
gnomad_df = pd.read_csv(gnomad_path, sep=";", low_memory=False)

print(f"Association results loaded: {associations_df.shape[0]:,} rows, {associations_df.shape[1]} columns")
print(f"gnomAD frequencies loaded:  {gnomad_df.shape[0]:,} rows, {gnomad_df.shape[1]} columns")

Association results loaded: 238,651 rows, 35 columns
gnomAD frequencies loaded:  840,229 rows, 22 columns


## Summary 1: Associations per gene and dataset

In [18]:
source_phenotype_order = [
    ("GWAS Catalog",      ""),
    ("HERMES",            "Heart failure"),
    ("CARDIoGRAMplusC4D", "Coronary artery disease"),
    ("CARDIoGRAMplusC4D", "Myocardial infarction"),
    ("FinnGen",           "Heart failure"),
    ("FinnGen",           "Coronary artery disease"),
    ("FinnGen",           "Myocardial infarction"),
]

column_labels = [
    "GWAS Catalog",
    "HERMES (HF)",
    "CARDIoGRAMplusC4D (CAD)",
    "CARDIoGRAMplusC4D (MI)",
    "FinnGen (HF)",
    "FinnGen (CAD)",
    "FinnGen (MI)",
]

all_genes = sorted(associations_df["official_gene_symbol"].dropna().unique())
summary_rows = []

for gene in all_genes:
    gene_df = associations_df[associations_df["official_gene_symbol"] == gene]
    row = {"Gene": gene}

    for (source, trait), label in zip(source_phenotype_order, column_labels):
        if source == "GWAS Catalog":
            count = int((gene_df["source_dataset"] == source).sum())
        else:
            count = int(
                ((gene_df["source_dataset"] == source) &
                 (gene_df["trait_name"] == trait)).sum()
            )
        row[label] = count

    row["Total"] = int(gene_df.shape[0])
    summary_rows.append(row)

associations_per_gene_df = pd.DataFrame(summary_rows).set_index("Gene")

total_row = associations_per_gene_df.sum(axis=0).rename("Total").to_frame().T
total_row.index.name = "Gene"
associations_per_gene_df = pd.concat([associations_per_gene_df, total_row])

print(f"Summary 1 — associations per gene and dataset: {associations_per_gene_df.shape}")
associations_per_gene_df

Summary 1 — associations per gene and dataset: (52, 8)


,GWAS Catalog,HERMES (HF),CARDIoGRAMplusC4D (CAD),CARDIoGRAMplusC4D (MI),FinnGen (HF),FinnGen (CAD),FinnGen (MI),Total
Gene,,,,,,,,
APOE,4426,376,390,371,879,879,879,8200
B2M,0,221,229,203,633,633,633,2552
C1QA,3,348,370,351,841,841,841,3595
C1QB,17,320,320,306,816,816,816,3411
C1QC,7,341,353,335,827,827,827,3517
C1R,57,255,251,242,720,720,720,2965
C2,315,401,415,385,1059,1059,1059,4693
CAPG,33,418,450,411,1018,1018,1018,4366
CCL2,50,312,328,303,805,805,805,3408


## Summary 2: Significant associations by p-value threshold

In [19]:
thresholds = {
    "genome_wide": 5e-8,
    "suggestive": 1e-5,
    "nominal": 0.05,
}

sig_rows = []

for gene in all_genes:
    gene_df = associations_df[associations_df["official_gene_symbol"] == gene]
    row = {"Gene": gene}

    for label, threshold in thresholds.items():
        row[f"p < {threshold:.0e}"] = int(
            (gene_df["p_value"] < threshold).sum()
        )

    row["Total associations"] = int(gene_df.shape[0])
    sig_rows.append(row)

significant_associations_df = pd.DataFrame(sig_rows).set_index("Gene")

total_row = significant_associations_df.sum(axis=0).rename("Total").to_frame().T
total_row.index.name = "Gene"
significant_associations_df = pd.concat([significant_associations_df, total_row])

print(f"Summary 2 — significant associations by p-value threshold: {significant_associations_df.shape}")
significant_associations_df

Summary 2 — significant associations by p-value threshold: (52, 4)


,p < 5e-08,p < 1e-05,p < 5e-02,Total associations
Gene,,,,
APOE,4355,4591,5433,8200
B2M,0,0,208,2552
C1QA,2,3,174,3595
C1QB,13,17,164,3411
C1QC,6,7,159,3517
C1R,57,57,404,2965
C2,301,336,1061,4693
CAPG,33,57,682,4366
CCL2,47,50,233,3408


## Summary 3: Top variants per gene

In [20]:
TOP_N = 10

top_variant_rows = []

for gene in all_genes:
    gene_df = associations_df[
        associations_df["official_gene_symbol"] == gene
    ].dropna(subset=["p_value", "rsID"])

    top = gene_df.nsmallest(TOP_N, "p_value")[[
        "official_gene_symbol",
        "rsID",
        "variant_id",
        "source_dataset",
        "trait_name",
        "p_value",
        "effect_size",
        "odds_ratio",
        "effect_allele",
        "other_allele",
        "allele_frequency",
        "location_relative_to_gene",
        "distance_from_gene",
        "region_assembly",
    ]]

    top_variant_rows.append(top)

top_variants_df = pd.concat(top_variant_rows, ignore_index=True)

print(f"Summary 3 — top {TOP_N} variants per gene: {top_variants_df.shape}")
top_variants_df.head(10)

Summary 3 — top 10 variants per gene: (510, 14)


,official_gene_symbol,rsID,variant_id,source_dataset,trait_name,p_value,effect_size,odds_ratio,effect_allele,other_allele,allele_frequency,location_relative_to_gene,distance_from_gene,region_assembly
0,APOE,rs429358,NaN,GWAS Catalog,Alzheimer disease; age at onset,0.0,NaN,NaN,?,NaN,NaN,NaN,NaN,NaN
1,APOE,rs7412,NaN,GWAS Catalog,Cholesterol:total lipids ratio; high density l...,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN
2,APOE,rs7412,NaN,GWAS Catalog,Cholesterol:total lipids ratio; high density l...,0.0,NaN,NaN,C,NaN,0.919396,NaN,NaN,NaN
3,APOE,rs7412,NaN,GWAS Catalog,Cholesteryl esters:total lipids ratio; Cholest...,0.0,NaN,NaN,T,NaN,0.074500,NaN,NaN,NaN
4,APOE,rs7412,NaN,GWAS Catalog,Cholesteryl esters:total lipids ratio; blood V...,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN
5,APOE,rs7412,NaN,GWAS Catalog,Cholesteryl esters:total lipids ratio; blood V...,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN
6,APOE,rs7412,NaN,GWAS Catalog,Cholesteryl esters:total lipids ratio; blood V...,0.0,NaN,NaN,T,NaN,0.075100,NaN,NaN,NaN
7,APOE,rs7412,NaN,GWAS Catalog,Cholesteryl esters:total lipids ratio; blood V...,0.0,NaN,NaN,C,NaN,0.919797,NaN,NaN,NaN
8,APOE,rs7412,NaN,GWAS Catalog,Cholesteryl esters:total lipids ratio; high de...,0.0,NaN,NaN,C,NaN,0.919396,NaN,NaN,NaN
9,APOE,rs7412,NaN,GWAS Catalog,Cholesteryl esters:total lipids ratio; high de...,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN


## Summary 4: Variants overlapping association results and gnomAD

In [21]:
assoc_rsids = set(associations_df["rsID"].dropna().unique())
gnomad_rsids = set(gnomad_df["rsID"].dropna().unique())

overlap_rows = []

for gene in all_genes:
    gene_assoc_rsids = set(
        associations_df.loc[
            associations_df["official_gene_symbol"] == gene, "rsID"
        ].dropna().unique()
    )
    gene_gnomad_rsids = set(
        gnomad_df.loc[
            gnomad_df["official_gene_symbol"] == gene, "rsID"
        ].dropna().unique()
    )

    overlapping = gene_assoc_rsids & gene_gnomad_rsids
    assoc_only = gene_assoc_rsids - gene_gnomad_rsids
    gnomad_only = gene_gnomad_rsids - gene_assoc_rsids

    overlap_rows.append({
        "Gene": gene,
        "Association rsIDs": len(gene_assoc_rsids),
        "gnomAD rsIDs": len(gene_gnomad_rsids),
        "Overlapping rsIDs": len(overlapping),
        "Association only": len(assoc_only),
        "gnomAD only": len(gnomad_only),
        "Overlap %": round(
            len(overlapping) / len(gene_assoc_rsids) * 100, 1
        ) if gene_assoc_rsids else 0.0,
    })

overlap_df = pd.DataFrame(overlap_rows).set_index("Gene")

total_row = overlap_df.sum(axis=0).rename("Total").to_frame().T
total_row.index.name = "Gene"
total_row["Overlap %"] = round(
    overlap_df["Overlapping rsIDs"].sum() /
    overlap_df["Association rsIDs"].sum() * 100, 1
)
overlap_df = pd.concat([overlap_df, total_row])

print(f"Summary 4 — gnomAD overlap: {overlap_df.shape}")
overlap_df

Summary 4 — gnomAD overlap: (52, 6)


,Association rsIDs,gnomAD rsIDs,Overlapping rsIDs,Association only,gnomAD only,Overlap %
Gene,,,,,,
APOE,898.0,1951.0,46.0,852.0,1905.0,5.1
B2M,623.0,1899.0,33.0,590.0,1866.0,5.3
C1QA,847.0,1335.0,32.0,815.0,1303.0,3.8
C1QB,821.0,2126.0,61.0,760.0,2065.0,7.4
C1QC,836.0,1461.0,43.0,793.0,1418.0,5.1
C1R,701.0,3406.0,62.0,639.0,3344.0,8.8
C2,1071.0,10368.0,331.0,740.0,10037.0,30.9
CAPG,1029.0,8087.0,254.0,775.0,7833.0,24.7
CCL2,801.0,561.0,9.0,792.0,552.0,1.1


## Summary 5: Functional consequence categories

In [22]:
consequence_df = (
    gnomad_df.groupby(["official_gene_symbol", "Functional_consequence"])
    .size()
    .reset_index(name="variant_count")
    .pivot(
        index="official_gene_symbol",
        columns="Functional_consequence",
        values="variant_count"
    )
    .fillna(0)
    .astype(int)
)

consequence_df.index.name = "Gene"

total_row = consequence_df.sum(axis=0).rename("Total").to_frame().T
total_row.index.name = "Gene"
consequence_df = pd.concat([consequence_df, total_row])

print(f"Summary 5 — functional consequence categories: {consequence_df.shape}")
consequence_df

Summary 5 — functional consequence categories: (53, 17)


Functional_consequence,3_prime_UTR_variant,5_prime_UTR_variant,frameshift_variant,inframe_deletion,inframe_insertion,intron_variant,missense_variant,non_coding_transcript_exon_variant,protein_altering_variant,splice_acceptor_variant,splice_donor_variant,splice_region_variant,start_lost,stop_gained,stop_lost,stop_retained_variant,synonymous_variant
Gene,,,,,,,,,,,,,,,,,
APOE,232,344,65,11,8,2240,859,4,0,5,10,43,3,57,3,1,366
B2M,760,139,4,0,1,4147,121,37,0,3,0,27,2,2,0,0,49
C1QA,317,227,41,5,2,1015,361,0,0,5,3,33,1,15,2,1,171
C1QB,296,460,16,5,1,3064,330,0,0,4,2,33,0,12,2,1,140
C1QC,557,351,28,5,6,1200,432,0,0,2,16,43,4,15,2,0,220
C1R,279,216,55,3,0,6209,984,8,0,23,11,99,10,53,2,1,409
C2,1458,1269,120,31,15,16310,1553,0,0,13,52,117,17,64,1,1,706
CAPG,1320,1176,66,13,5,12455,740,52,1,26,44,176,5,33,4,2,299
CCL2,743,64,5,0,0,934,115,19,0,3,0,31,1,6,0,1,53


In [23]:
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    associations_per_gene_df.to_excel(
        writer,
        sheet_name="1. Associations per gene",
    )
    significant_associations_df.to_excel(
        writer,
        sheet_name="2. Significant by p-value",
    )
    top_variants_df.to_excel(
        writer,
        sheet_name="3. Top variants per gene",
        index=False,
    )
    overlap_df.to_excel(
        writer,
        sheet_name="4. gnomAD overlap",
    )
    consequence_df.to_excel(
        writer,
        sheet_name="5. Functional consequences",
    )